In [ ]:
import json

In [ ]:
with open("../../data/processed/cleaned_users.json", "r") as f:
    users = json.load(f)

# filtered users (80% of all users by total_interactions count -> kick top 20% of users with most interactions to focus on more "average" users)
interactions = [u["total_interactions"] for u in users]
threshold = sorted(interactions, reverse=True)[int(len(interactions) * 0.2)]
filtered_users = [u for u in users if u["total_interactions"] < threshold]

In [ ]:
# simply predict likes based on the average like ratio per user
correct_predictions = 0
total_predictions = 0

for user in filtered_users:

    if user["like_ratio"] >= 0.5:
        correct_predictions += user["like_count"]
    else:
        correct_predictions += user["dislike_count"]
        
    total_predictions += user["total_interactions"]
print(f"Accuracy: {correct_predictions / total_predictions:.4f}")

In [ ]:
for user in users:
    like_ratio = user["like_ratio"]
    if like_ratio >= 0.5:
        correct_predictions += user["like_count"]
    else:
        correct_predictions += user["dislike_count"]
        
    total_predictions += user["total_interactions"]

print(f"Accuracy: {correct_predictions / total_predictions:.4f}")

In [ ]:
# Create Task Dataset (train/test split) for both user datasets.
# The Dataset is Next Like Prediction starting with 3 interactions and predicting the 4th interaction (like/dislike).
# We will create two datasets: one with all users and one with filtered users (top 20% of users by total_interactions count).
import random

def create_dataset(users, filename):
    dataset = []
    for user in users:
        interactions = user["interactions"]["likes"] + user["interactions"]["dislikes"]
        for interaction in interactions:
            interaction["type"] = "like" if interaction in user["interactions"]["likes"] else "dislike"
        
        # sort by key "date"
        interactions.sort(key=lambda x: x["date"])

        if len(interactions) < 4:
            continue

        input_data = [{"text": interactions[i]["text"], "interaction_id": interactions[i]["id"], "label": 1 if interactions[i]["type"] == "like" else 0, "date": interactions[i]["date"], "language": interactions[i]["lang"]} for i in range(3)]
        output_data = [] 
        for i in range(3, len(interactions)):
            output_data.append({"text": interactions[i]["text"], "label": 1 if interactions[i]["type"] == "like" else 0, "interaction_id": interactions[i]["id"], "round": i, "date": interactions[i]["date"], "language": interactions[i]["lang"]})

        dataset.append({"user_id": user["user_id"], "input": input_data, "output": output_data})
    with open(filename, "w") as f:
        json.dump(dataset, f, indent=4)

random.seed(42)

test_users = random.sample(users, int(len(users) * 0.2))
train_users = [u for u in users if u not in test_users]

create_dataset(train_users, "../../data/processed/next-like-prediction-train.json")
create_dataset(test_users, "../../data/processed/next-like-prediction-test.json")


test_filtered_users = random.sample(filtered_users, int(len(filtered_users) * 0.2))
train_filtered_users = [u for u in filtered_users if u not in test_filtered_users]

create_dataset(train_filtered_users, "../../data/processed/next-like-prediction-train-filtered.json")
create_dataset(test_filtered_users, "../../data/processed/next-like-prediction-test-filtered.json")